<a href="https://colab.research.google.com/github/reddytian/dl-learning/blob/main/day5_transfer_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 5 — Transfer Learning on CIFAR-10 (ResNet-18, ImageNet pretrained)

Day 4 trained a CNN from scratch and reached **76.6% test accuracy** with a +21 point
train/test gap. The diagnosis was that the binding constraint is *data variety*, not
model capacity — 666,538 parameters against 50,000 images, with training accuracy at
0.976.

Transfer learning attacks that constraint directly. A ResNet-18 pretrained on
ImageNet-1k learned its features from 1.2 million images across 1,000 classes. Its
early layers are edge, texture and colour detectors; its middle layers are parts and
motifs. Only the final layer is specific to ImageNet's classes. Reusing the first
three imports 1.2 million images' worth of visual priors at no data-collection cost.

**Two modes, used in sequence:**

| | Feature extraction (stage A) | Fine-tuning (stage B) |
|---|---|---|
| backbone | frozen | unfrozen, low LR |
| trainable params | ~5k | ~11M |
| speed | fast | slower |
| ceiling | lower | higher |
| overfitting risk | low | real |

**Baseline to beat: 0.766 test accuracy** (from-scratch CNN, 15 epochs, Day 4).

This notebook is self-contained — it re-defines `device`, `loss_fn` and `accuracy()`
rather than assuming a Day 4 runtime.

## 0. Setup

Set the Colab runtime to a GPU before running: **Runtime > Change runtime type > T4 GPU**.

In [1]:
import time
import torch, torch.nn as nn
import torchvision, torchvision.transforms as T
from torch.utils.data import DataLoader
from torchvision.models import resnet18, ResNet18_Weights

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

loss_fn = nn.CrossEntropyLoss()

def accuracy(model, dl):
    # model.eval() matters here: ResNet is full of BatchNorm, which behaves
    # differently in training vs inference mode. Omitting it silently corrupts
    # the numbers -- no error, just wrong accuracy.
    model.eval(); correct = total = 0
    with torch.no_grad():
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            correct += (model(xb).argmax(1) == yb).sum().item()
            total += yb.size(0)
    return correct / total

BASELINE = 0.766   # Day 4, from-scratch CNN, 15 epochs

cuda


## 1. Inspect the pretrained model

Pretrained weights come with a **preprocessing contract**: the input size and channel
statistics the network was trained with. `weights.transforms()` prints it. Feeding
differently-normalized data produces no error and quietly degrades the features — the
most common transfer-learning mistake.

In [2]:
weights = ResNet18_Weights.DEFAULT          # ImageNet-1k pretrained
net = resnet18(weights=weights)

print(net.fc)                                # the ImageNet classifier head
print("total params:", sum(p.numel() for p in net.parameters()))
print(weights.transforms())                  # the required preprocessing

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 177MB/s]


Linear(in_features=512, out_features=1000, bias=True)
total params: 11689512
ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


## 2. Data, matched to the contract

Note the normalization statistics are **ImageNet's, not CIFAR-10's**. Day 4 used
CIFAR-10's own channel statistics, which was correct there and would be wrong here.

Upsampling 32x32 to 224x224 invents no information — it is pure interpolation. It is
done because the pretrained network's receptive fields and stride schedule assume
that input scale. The cost is 49x the pixels, so expect roughly 2-3 minutes per epoch
instead of Day 4's 27 seconds.

In [3]:
imnet = ((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))   # ImageNet stats

tf_tl_train = T.Compose([
    T.Resize(224), T.RandomHorizontalFlip(),
    T.ToTensor(), T.Normalize(*imnet),
])
tf_tl_test = T.Compose([
    T.Resize(224),
    T.ToTensor(), T.Normalize(*imnet),
])

tl_train = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=tf_tl_train)
tl_test  = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=tf_tl_test)

tl_train_dl = DataLoader(tl_train, batch_size=128, shuffle=True,  num_workers=2, pin_memory=True)
tl_test_dl  = DataLoader(tl_test,  batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

xb, yb = next(iter(tl_train_dl))
print(xb.shape)     # expect [128, 3, 224, 224]

100%|██████████| 170M/170M [16:43<00:00, 170kB/s]


torch.Size([128, 3, 224, 224])


## 3. Stage A — freeze the backbone, train a new head

`requires_grad = False` stops autograd recording those parameters, so `backward()`
computes no gradients for them and they cannot move. Replacing `.fc` creates a fresh
`Linear` whose parameters default to `requires_grad=True` — that is what trains.

Passing only the trainable parameters to the optimizer is not cosmetic: Adam keeps
momentum state per parameter, so handing it frozen ones wastes memory and leaves
stale state attached if they are later unfrozen.

**Subtlety worth knowing:** `model.train()` still updates BatchNorm's running mean and
variance even for frozen layers, because those are buffers rather than parameters.
Freezing weights does not freeze statistics. Usually harmless, occasionally helpful,
and a genuine practitioner detail.

In [4]:
model_tl = resnet18(weights=ResNet18_Weights.DEFAULT)

for p in model_tl.parameters():          # freeze everything
    p.requires_grad = False

model_tl.fc = nn.Linear(512, 10)         # new head, trainable by default
model_tl = model_tl.to(device)

trainable = [p for p in model_tl.parameters() if p.requires_grad]
print("trainable params:", sum(p.numel() for p in trainable), "of",
      sum(p.numel() for p in model_tl.parameters()))

opt_tl = torch.optim.Adam(trainable, lr=1e-3)    # ONLY the trainable params
hist_tl = {"test_acc": []}

for ep in range(3):
    t0 = time.time(); model_tl.train()
    for xb, yb in tl_train_dl:
        xb, yb = xb.to(device), yb.to(device)
        loss = loss_fn(model_tl(xb), yb)
        opt_tl.zero_grad(); loss.backward(); opt_tl.step()
    te = accuracy(model_tl, tl_test_dl)
    hist_tl["test_acc"].append(te)
    print(f"[stage A] ep {ep+1}  test {te:.3f}  "
          f"(vs baseline {BASELINE:.3f})  ({time.time()-t0:.0f}s)")

trainable params: 5130 of 11181642
[stage A] ep 1  test 0.779  (vs baseline 0.766)  (110s)
[stage A] ep 2  test 0.796  (vs baseline 0.766)  (104s)
[stage A] ep 3  test 0.804  (vs baseline 0.766)  (103s)


Train accuracy is deliberately not measured in stage A — it would cost a full extra
pass over 50,000 upsampled images per epoch. The generalization gap gets measured
properly in stage B.

**Prediction on record (to be checked against the result):** 80-85% test accuracy
after 3 epochs, beating the 15-epoch from-scratch baseline while training ~5,000
parameters instead of 666,538.

## 4. Stage B — fine-tuning

*To be added once stage A results are in.* Stage B unfreezes the backbone and trains
it at a much lower learning rate; the LR choice is the whole game, since too high
destroys the pretrained weights that stage A just demonstrated the value of.

## Note — transfer learning on satellite imagery

ImageNet transfer works less well on overhead imagery than these CIFAR-10 numbers
suggest, and it is worth being precise about why:

- **No consistent object scale or orientation.** ImageNet is ground-level photography
  with a canonical "up" and objects at predictable scales. Nadir-view imagery has
  neither.
- **Different texture statistics.** Land cover does not look like household objects.
- **Channel mismatch.** A 13-band Sentinel-2 stack does not fit a 3-channel first
  conv layer. Standard workarounds exist (replicate or average the pretrained RGB
  filters across the extra bands, or re-initialize just that layer), but it is a real
  adaptation step, not a drop-in.

Transfer still helps, usually substantially. But remote-sensing-specific pretrained
backbones — SSL4EO, Prithvi, models trained on BigEarthNet — start from satellite
data and typically transfer better to satellite tasks. Worth using in the
segmentation project in weeks 5-6.

## Summary

*To be completed after stage A and stage B are run.*

Structure of the comparison being built:

| model | trainable params | epochs | test accuracy |
|---|---|---|---|
| Day 4 from-scratch CNN | 666,538 | 15 | 0.766 |
| Stage A, frozen backbone | ~5,130 | 3 | *pending* |
| Stage B, fine-tuned | ~11.2M | *pending* | *pending* |